# 02 - Generate Chain-of-Thought Responses

Uses vLLM for fast batched generation of COTs from Qwen3-4B on a 128-problem subset of GSM8K.

**Fully resumable** - caches one JSON file per problem in `cache/cots/`. Re-run all cells to continue from where you left off.

In [1]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [ ]:
import json
import re
import traceback
from tqdm.auto import tqdm
from lib.data import load_gsm8k, extract_predicted_answer, build_generation_messages

In [ ]:
# Install vLLM if needed
!pip install -q vllm

In [ ]:
# Load dataset (128 subset)
dataset = load_gsm8k()[:SUBSET_SIZE]
print(f"Using first {SUBSET_SIZE} GSM8K problems")

# Resume logic
done_ids = {int(p.stem) for p in COT_CACHE.glob("*.json")}
todo = [ex for ex in dataset if ex["problem_id"] not in done_ids]
print(f"Resuming: {len(done_ids)} done, {len(todo)} remaining")

In [ ]:
# Load model with vLLM
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = LLM(model=MODEL_NAME, dtype="bfloat16", max_model_len=4096)

sampling_params = SamplingParams(
    max_tokens=MAX_COT_TOKENS,
    temperature=0.0,  # greedy
)
print(f"vLLM loaded {MODEL_NAME}")

In [ ]:
# Batched COT generation with vLLM
if not todo:
    print("All problems already cached, skipping generation.")
else:
    # Build prompts
    prompts = []
    for ex in todo:
        messages = build_generation_messages(ex["question"])
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)

    print(f"Generating COTs for {len(prompts)} problems...")
    outputs = llm.generate(prompts, sampling_params)

    # Process results and cache
    correct = 0
    total = 0

    for ex, output in zip(todo, outputs):
        try:
            generated = output.outputs[0].text

            # Extract thinking content from <think>...</think> tags
            think_match = re.search(r"<think>(.*?)</think>", generated, re.DOTALL)
            cot_text = think_match.group(1).strip() if think_match else generated.strip()

            # Extract the answer portion (after </think>)
            if think_match:
                answer_portion = generated[think_match.end():].strip()
            else:
                answer_portion = generated.strip()

            predicted_answer = extract_predicted_answer(answer_portion)

            cache_entry = {
                "problem_id": ex["problem_id"],
                "question": ex["question"],
                "gold_answer": ex["gold_answer"],
                "cot_text": cot_text,
                "full_response": generated,
                "answer_portion": answer_portion,
                "predicted_answer": predicted_answer,
                "error": None,
            }
            if predicted_answer == ex["gold_answer"]:
                correct += 1
            total += 1
        except Exception:
            cache_entry = {
                "problem_id": ex["problem_id"],
                "question": ex["question"],
                "gold_answer": ex["gold_answer"],
                "cot_text": None,
                "full_response": None,
                "answer_portion": None,
                "predicted_answer": None,
                "error": traceback.format_exc(),
            }

        (COT_CACHE / f"{ex['problem_id']}.json").write_text(json.dumps(cache_entry))

    if total > 0:
        print(f"Batch accuracy: {correct}/{total} = {correct/total:.1%}")

In [ ]:
# Aggregate all cache files into results/cots.jsonl
all_cots = []
errors = 0
correct = 0

for p in sorted(COT_CACHE.glob("*.json"), key=lambda x: int(x.stem)):
    entry = json.loads(p.read_text())
    all_cots.append(entry)
    if entry["error"]:
        errors += 1
    elif entry["predicted_answer"] == entry["gold_answer"]:
        correct += 1

# Write aggregated results
output_path = RESULTS_DIR / "cots.jsonl"
with open(output_path, "w") as f:
    for entry in all_cots:
        f.write(json.dumps(entry) + "\n")

valid = len(all_cots) - errors
print(f"Total: {len(all_cots)} | Valid: {valid} | Errors: {errors}")
print(f"Normal accuracy (ceiling): {correct}/{valid} = {correct/valid:.1%}" if valid > 0 else "No valid results")
print(f"Saved to {output_path}")

In [ ]:
# Show a few examples
for entry in all_cots[:3]:
    if entry["error"]:
        continue
    print(f"--- Problem {entry['problem_id']} ---")
    print(f"Q: {entry['question'][:100]}...")
    print(f"COT (first 200 chars): {entry['cot_text'][:200]}...")
    print(f"Predicted: {entry['predicted_answer']} | Gold: {entry['gold_answer']}")
    print()